# Notebook 2: Model Training

This notebook walks through training all three Arabic ASR models:
1. CNN+LSTM (custom, trained from scratch)
2. Whisper (zero-shot inference — no fine-tuning needed)
3. Wav2Vec 2.0 (inference with pretrained Arabic model)

> **Tip:** For long training runs, use the CLI script instead:
> ```bash
> python training/train_cnn_lstm.py --config configs/config.yaml
> ```

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import yaml
import numpy as np

# Load config
with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

device = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')
print(f'Config loaded. Dataset: {config["data"]["dataset_name"]}')

## 1. Load Data

In [ ]:
from data.dataset import get_dataloaders

# Use smaller subset for notebook exploration
config['data']['max_train_samples'] = 500
config['data']['max_eval_samples'] = 100

train_loader, val_loader, test_loader, vocab = get_dataloaders(config)
print(f'Vocab size: {len(vocab)}')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

# Inspect one batch
batch = next(iter(train_loader))
print(f'\nBatch mel shape:    {batch["mel"].shape}  (B, 1, n_mels, T)')
print(f'Batch tokens shape: {batch["tokens"].shape}  (B, max_seq_len)')
print(f'Sample transcript:  {batch["transcripts"][0]}')

## 2. CNN+LSTM Model — Architecture

In [ ]:
from models.cnn_lstm_asr import build_model

model = build_model(vocab_size=len(vocab), config=config).to(device)
print(f'Model parameters: {model.count_parameters():,}')
print()
print(model)

In [ ]:
# Test forward pass
model.eval()
with torch.no_grad():
    mel = batch['mel'].to(device)
    log_probs = model(mel)
print(f'Output shape: {log_probs.shape}  (B, T, vocab_size)')
print(f'Log-prob range: [{log_probs.min():.2f}, {log_probs.max():.2f}]')

## 3. Train CNN+LSTM (Mini Run)

In [ ]:
import torch.optim as optim
from tqdm.notebook import tqdm

# Short training run: 3 epochs for demonstration
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
model.train()

N_EPOCHS = 3
train_losses = []

for epoch in range(1, N_EPOCHS + 1):
    epoch_loss = 0.0
    n_batches = 0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{N_EPOCHS}'):
        mel = batch['mel'].to(device)
        tokens = batch['tokens'].to(device)
        target_lengths = batch['target_lengths'].to(device)

        optimizer.zero_grad()
        log_probs = model(mel)
        ctc_lens = torch.full((mel.shape[0],), log_probs.shape[1], dtype=torch.long, device=device)

        loss = model.ctc_loss(log_probs, tokens, ctc_lens, target_lengths)
        if torch.isfinite(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

    avg = epoch_loss / max(n_batches, 1)
    train_losses.append(avg)
    print(f'  Epoch {epoch} | Loss: {avg:.4f}')

print('Training complete (demo run).')

## 4. Whisper — Zero-Shot Arabic Inference

In [ ]:
from models.whisper_asr import WhisperASR

# Use 'tiny' for fast notebook demo; change to 'medium' for better accuracy
whisper = WhisperASR(model_size='tiny', language='ar')

# Transcribe a test sample
from data.dataset import load_common_voice_arabic
test_ds = load_common_voice_arabic('test', max_samples=5)

for sample in list(test_ds)[:3]:
    audio = sample['audio']['array'].astype(np.float32)
    ref = sample['sentence']
    hyp = whisper.transcribe_array(audio)
    print(f'REF: {ref}')
    print(f'HYP: {hyp}')
    print()

## 5. Wav2Vec 2.0 — Arabic Inference

In [ ]:
from models.wav2vec_asr import Wav2Vec2ASR

wav2vec = Wav2Vec2ASR()

for sample in list(test_ds)[:3]:
    audio = sample['audio']['array'].astype(np.float32)
    ref = sample['sentence']
    hyp = wav2vec.transcribe_array(audio)
    print(f'REF: {ref}')
    print(f'HYP: {hyp}')
    print()